In [1]:
#Extracting the feature for 10 subjects all 64 channels

import os
import numpy as np
import mne
from scipy.signal import welch

# ==========================================================
# CONFIG
# ==========================================================
SUBJECTS = [
    "S001", "S002", "S003", "S004", "S005",
    "S006", "S007", "S008", "S009", "S010",
]
EPOCH_DIR = "/Users/prajwalnara/Documents/EEG-motor-imagery/data/processed/epochs"
FEATURE_DIR = "/Users/prajwalnara/Documents/EEG-motor-imagery/data/features/band_power_10sub_full64"
os.makedirs(FEATURE_DIR, exist_ok=True)

REQUIRED_EVENT_NAMES = {"rest", "left", "right"}

# Fixed explicitly (not adapted per-epoch) so frequency resolution is
# identical across every subject — required for features to be
# comparable when pooled together in Stage 3.
NPERSEG = 256


# ==========================================================
# FEATURE EXTRACTION
# ==========================================================
def extract_bandpower_features(epochs, nperseg=NPERSEG):
    data = epochs.get_data()
    sfreq = epochs.info["sfreq"]

    if data.shape[-1] < nperseg:
        raise ValueError(
            f"Epoch length ({data.shape[-1]} samples) shorter than "
            f"nperseg ({nperseg})."
        )

    features = []
    for epoch in data:
        epoch_features = []
        for channel in epoch:
            freqs, psd = welch(channel, fs=sfreq, nperseg=nperseg)
            mu_mask = (freqs >= 8) & (freqs <= 12)
            beta_mask = (freqs >= 13) & (freqs <= 30)
            mu_power = np.trapezoid(psd[mu_mask], freqs[mu_mask])
            beta_power = np.trapezoid(psd[beta_mask], freqs[beta_mask])
            epoch_features.extend([mu_power, beta_power])
        features.append(epoch_features)
    return np.array(features)


# ==========================================================
# PROCESS SUBJECTS
# ==========================================================
succeeded, failed = [], []

for subject in SUBJECTS:
    print(f"\nProcessing {subject}")
    subject_epoch_dir = os.path.join(EPOCH_DIR, subject)

    try:
        r04_path = os.path.join(subject_epoch_dir, f"{subject}R04_epo.fif")
        r08_path = os.path.join(subject_epoch_dir, f"{subject}R08_epo.fif")
        r12_path = os.path.join(subject_epoch_dir, f"{subject}R12_epo.fif")

        for p in (r04_path, r08_path, r12_path):
            if not os.path.exists(p):
                raise FileNotFoundError(p)

        r04 = mne.read_epochs(r04_path, preload=True, verbose=False)
        r08 = mne.read_epochs(r08_path, preload=True, verbose=False)
        r12 = mne.read_epochs(r12_path, preload=True, verbose=False)

        # Verify event_id mapping is consistent across this subject's
        # own runs AND matches the expected rest/left/right names.
        # Don't assume the integer codes (1/2/3) match across subjects —
        # save the actual mapping per subject instead.
        for ep, name in [(r04, "R04"), (r08, "R08"), (r12, "R12")]:
            if set(ep.event_id.keys()) != REQUIRED_EVENT_NAMES:
                raise ValueError(
                    f"{subject} {name}: unexpected event names "
                    f"{set(ep.event_id.keys())}, expected {REQUIRED_EVENT_NAMES}"
                )

        if not (r04.event_id == r08.event_id == r12.event_id):
            raise ValueError(
                f"{subject}: event_id mapping differs between runs "
                f"(R04={r04.event_id}, R08={r08.event_id}, R12={r12.event_id})"
            )

        event_id_map = r04.event_id

        train_epochs = mne.concatenate_epochs([r04, r08])

        if len(train_epochs) == 0 or len(r12) == 0:
            raise ValueError(f"{subject}: zero epochs after concatenation")

        X_train = extract_bandpower_features(train_epochs)
        X_test = extract_bandpower_features(r12)

        y_train = train_epochs.events[:, -1]
        y_test = r12.events[:, -1]

        # Subject ID tagged alongside every row — needed for subject-level
        # train/test splitting once all subjects are pooled together.
        subject_id_train = np.full(len(y_train), subject)
        subject_id_test = np.full(len(y_test), subject)

        save_dir = os.path.join(FEATURE_DIR, subject)
        os.makedirs(save_dir, exist_ok=True)

        np.save(os.path.join(save_dir, "X_train.npy"), X_train)
        np.save(os.path.join(save_dir, "y_train.npy"), y_train)
        np.save(os.path.join(save_dir, "X_test.npy"), X_test)
        np.save(os.path.join(save_dir, "y_test.npy"), y_test)
        np.save(os.path.join(save_dir, "subject_id_train.npy"), subject_id_train)
        np.save(os.path.join(save_dir, "subject_id_test.npy"), subject_id_test)
        np.save(
            os.path.join(save_dir, "event_id_map.npy"),
            event_id_map,
            allow_pickle=True,
        )

        print(
            f"  OK — X_train {X_train.shape}, X_test {X_test.shape}, "
            f"event_id={event_id_map}"
        )
        succeeded.append(subject)

    except Exception as e:
        print(f"  FAILED {subject}: {e}")
        failed.append((subject, str(e)))
        continue

# ==========================================================
# SUMMARY
# ==========================================================
print("\n========== SUMMARY ==========")
print(f"Succeeded: {len(succeeded)}/{len(SUBJECTS)} -> {succeeded}")
if failed:
    print(f"Failed: {len(failed)}")
    for subj, err in failed:
        print(f"  {subj}: {err}")
else:
    print("No failures.")
print("\nDone.")


Processing S001
Not setting metadata
60 matching events found
No baseline correction applied


/var/folders/2p/pvb41hns2szdm00nww51m7r40000gn/T/ipykernel_77351/4291586320.py:95: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  train_epochs = mne.concatenate_epochs([r04, r08])


  OK — X_train (60, 128), X_test (30, 128), event_id={'rest': 1, 'left': 2, 'right': 3}

Processing S002
Not setting metadata
60 matching events found
No baseline correction applied


/var/folders/2p/pvb41hns2szdm00nww51m7r40000gn/T/ipykernel_77351/4291586320.py:95: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  train_epochs = mne.concatenate_epochs([r04, r08])


  OK — X_train (60, 128), X_test (30, 128), event_id={'rest': 1, 'left': 2, 'right': 3}

Processing S003
Not setting metadata
60 matching events found
No baseline correction applied


...

  OK — X_train (60, 128), X_test (30, 128), event_id={'rest': 1, 'left': 2, 'right': 3}

Processing S009
Not setting metadata
60 matching events found
No baseline correction applied


/var/folders/2p/pvb41hns2szdm00nww51m7r40000gn/T/ipykernel_77351/4291586320.py:95: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  train_epochs = mne.concatenate_epochs([r04, r08])


  OK — X_train (60, 128), X_test (30, 128), event_id={'rest': 1, 'left': 2, 'right': 3}

Processing S010
Not setting metadata
60 matching events found
No baseline correction applied


/var/folders/2p/pvb41hns2szdm00nww51m7r40000gn/T/ipykernel_77351/4291586320.py:95: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  train_epochs = mne.concatenate_epochs([r04, r08])


  OK — X_train (60, 128), X_test (30, 128), event_id={'rest': 1, 'left': 2, 'right': 3}

========== SUMMARY ==========
Succeeded: 10/10 -> ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010']
No failures.

Done.
